# Third-person framing ablation — ANES 2020


In [1]:
import os, zlib
import pandas as pd
import numpy as np
from collections import Counter
from scipy.stats import entropy

In [2]:
# the four ANES 2020 items that have a third-person variant)
FOUR_QIDS = ["V201324", "V202234", "V202257", "V202371"]
name_dict = {
    "V201324": "Current Economy",     # non-directional -> JSD only, no D
    "V202234": "Refugee Allowing",
    "V202257": "Income Inequality",
    "V202371": "Race Diversity",
}
name_to_qid = {v: k for k, v in name_dict.items()}
FIVE_POINT = {"V201324", "V202332"}


def option_codes(qid):
    return list(range(1, 6)) if qid in FIVE_POINT else list(range(1, 4))

In [3]:
desirability_spec = {
    "Current Economy":   {"kind": None,          "directional": False},                    # V201324
    "Refugee Allowing":  {"kind": "categorical", "desirable": [1], "directional": True},   # V202234 1=Favor
    "Income Inequality": {"kind": "categorical", "desirable": [1], "directional": True},   # V202257 1=Favor reducing
    "Race Diversity":    {"kind": "categorical", "desirable": [1], "directional": True},    # V202371 1=Better
}


def build_weights(entry, codes):
    codes = sorted(codes)
    if not entry.get("directional", True):
        return None
    kind = entry.get("kind")
    if kind == "categorical":
        desirable = entry.get("desirable")
        if isinstance(desirable, int):
            desirable = [desirable]
        desirable = set(desirable)
        assert desirable, "categorical item needs at least one desirable code"
        assert desirable <= set(codes), (
            f"desirable {sorted(desirable)} not within option codes {codes}; "
            f"check the CSV code->text mapping"
        )
        w = {c: (1.0 if c in desirable else 0.0) for c in codes}
        return w, 1.0
    if kind == "ordinal":
        lo, hi = codes[0], codes[-1]
        de = entry["desirable_end"]
        assert de in (lo, hi), f"desirable_end {de} must be endpoint {lo} or {hi}"
        w = {c: float(c) for c in codes} if de == hi else {c: float(hi + lo - c) for c in codes}
        return w, float(hi - lo)
    raise ValueError(f"bad 'kind': {kind!r}")

In [4]:
def get_counts(path_prefix, qids):
    value_list = []
    for qid in qids:
        fp = path_prefix + qid + ".csv"
        if not os.path.exists(fp):
            raise FileNotFoundError(
                f"expected results file not found:\n  {fp}\n"
                f"check the model directory and the condition folder name."
            )
        df = pd.read_csv(fp)
        df["Response"] = pd.to_numeric(df["Response"], errors="coerce")
        vc = Counter(df["Response"].dropna())
        codes = option_codes(qid)
        answers = {c: vc[c] for c in codes}
        value_list.append({name_dict[qid]: answers})
    return value_list


def get_human_counts(anes_csv, qids):
    if not os.path.exists(anes_csv):
        raise FileNotFoundError(f"ANES human CSV not found:\n  {anes_csv}")
    anes_df = pd.read_csv(anes_csv)
    human_values = []
    for qid in qids:
        hc = Counter(anes_df[qid].dropna())
        codes = option_codes(qid)
        answers = {c: hc[c] for c in codes}
        human_values.append({name_dict[qid]: answers})
    return human_values

In [5]:
def jsd_divergence_between_sets(set1, set2, base=2):
    jsd_results = {}
    d1 = {list(d)[0]: list(d.values())[0] for d in set1}
    d2 = {list(d)[0]: list(d.values())[0] for d in set2}
    for q in d1:
        if q not in d2:
            continue
        pc, qc = d1[q], d2[q]
        p = np.array([pc.get(k, 0) for k in pc], dtype=float)
        qq = np.array([qc.get(k, 0) for k in pc], dtype=float)
        p /= p.sum(); qq /= qq.sum()
        eps = 1e-10
        p = np.clip(p, eps, 1); qq = np.clip(qq, eps, 1)
        m = 0.5 * (p + qq)
        jsd_results[q] = 0.5 * (entropy(p, m, base=base) + entropy(qq, m, base=base))
    return jsd_results


def to_dict(count_list):
    return {list(d)[0]: list(d.values())[0] for d in count_list}


def normalize(counts, codes):
    total = sum(counts.get(c, 0) for c in codes)
    if total == 0:
        return {c: 0.0 for c in codes}, 0
    return {c: counts.get(c, 0) / total for c in codes}, total


def desirability_score(counts, weights, codes):
    p, _ = normalize(counts, codes)
    return sum(weights[c] * p[c] for c in codes)

In [6]:
def gap_analysis(model_base_list, model_cond_list, human_list):
    mb, mc, hu = to_dict(model_base_list), to_dict(model_cond_list), to_dict(human_list)
    jsd_base = jsd_divergence_between_sets(model_base_list, human_list)
    jsd_cond = jsd_divergence_between_sets(model_cond_list, human_list)
    rows = []
    for name in mb:
        codes = option_codes(name_to_qid[name])
        built = build_weights(desirability_spec[name], codes)
        jb, jc = jsd_base[name], jsd_cond[name]
        if built is None:                       # non-directional: JSD only
            rows.append({"item": name, "directional": False,
                         "D_base": None, "D_cond": None, "delta_absD": None,
                         "delta_jsd": jc - jb, "jsd_base": jb, "jsd_cond": jc})
            continue
        weights, norm = built
        S_h = desirability_score(hu[name], weights, codes)
        D_base = (desirability_score(mb[name], weights, codes) - S_h) / norm
        D_cond = (desirability_score(mc[name], weights, codes) - S_h) / norm
        rows.append({"item": name, "directional": True,
                     "D_base": D_base, "D_cond": D_cond,
                     "delta_absD": abs(D_cond) - abs(D_base),
                     "delta_jsd": jc - jb, "jsd_base": jb, "jsd_cond": jc})
    return rows

In [7]:
# Bootstrap machinery   
N_BOOT = 10_000
SEED = 20250701


def _cvec(counts, codes):
    return np.array([counts.get(c, 0) for c in codes], dtype=float)


def _resample(cvec, rng, n_rep):
    n = cvec.sum()
    if n == 0:
        return np.zeros((n_rep, len(cvec)))
    return rng.multinomial(int(n), cvec / n, size=n_rep).astype(float)


def _S(cmat, w):
    tot = cmat.sum(axis=1, keepdims=True)
    tot = np.where(tot == 0, np.nan, tot)
    return (cmat / tot) @ w


def boot_p_two_sided(delta):
    n = len(delta)
    k = min(int(np.sum(delta <= 0)), int(np.sum(delta >= 0)))
    return min(1.0, 2.0 * (1 + k) / (n + 1))


def boot_p_one_sided(x, tail):
    n = len(x)
    k = int(np.sum(x >= 0)) if tail == "less" else int(np.sum(x <= 0))
    return (1 + k) / (n + 1)


def bh_adjust(p):
    p = np.asarray(p, dtype=float)
    n = len(p)
    if n == 0:
        return p
    order = np.argsort(p)
    ranked = p[order] * n / np.arange(1, n + 1)
    adj = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(adj, 0, 1)
    return out


def _jsd_rows(Pm, Qm, base=2.0):
    p = Pm / Pm.sum(axis=1, keepdims=True)
    q = Qm / Qm.sum(axis=1, keepdims=True)
    p = np.clip(p, 1e-10, 1.0); q = np.clip(q, 1e-10, 1.0)
    p = p / p.sum(axis=1, keepdims=True); q = q / q.sum(axis=1, keepdims=True)
    m = 0.5 * (p + q); m = m / m.sum(axis=1, keepdims=True)
    kl = lambda a: np.sum(a * np.log(a / m), axis=1)
    return 0.5 * (kl(p) + kl(q)) / np.log(base)


def _jsd_point(counts_a, counts_b, codes):
    A = _cvec(counts_a, codes)[None, :]
    B = _cvec(counts_b, codes)[None, :]
    return float(_jsd_rows(A, B)[0])


def _item_rng(name, seed):
    return np.random.default_rng([seed, zlib.crc32(name.encode())])


def bootstrap_jsd(baseline, conditions, human, n_rep=N_BOOT, seed=SEED):
    """conditions: {cname: [{name:{code:count}}, ...]}  (already item-subset)."""
    hu, mb = to_dict(human), to_dict(baseline)
    prepared = {c: to_dict(v) for c, v in conditions.items()}
    base_rows, cond_rows = [], []
    per_item_delta, per_item_base, per_item_cond = {}, {}, {}
    for name in mb:
        codes = option_codes(name_to_qid[name])
        rng = _item_rng(name, seed)
        H = _resample(_cvec(hu[name], codes), rng, n_rep)
        B = _resample(_cvec(mb[name], codes), rng, n_rep)
        J_b = _jsd_rows(B, H)
        J_b_point = _jsd_point(mb[name], hu[name], codes)
        base_rows.append({"item": name, "jsd_base": J_b_point,
                          "ci_lo": float(np.percentile(J_b, 2.5)),
                          "ci_hi": float(np.percentile(J_b, 97.5)),
                          "n_silicon": int(_cvec(mb[name], codes).sum())})
        for cname, cd in prepared.items():
            if name not in cd:
                continue
            C = _resample(_cvec(cd[name], codes), rng, n_rep)
            J_c = _jsd_rows(C, H)
            d = J_c - J_b
            J_c_point = _jsd_point(cd[name], hu[name], codes)
            per_item_delta.setdefault(cname, []).append(d)
            per_item_base.setdefault(cname, []).append(J_b)
            per_item_cond.setdefault(cname, []).append(J_c)
            cond_rows.append({"item": name, "condition": cname,
                              "jsd_base": J_b_point, "jsd_cond": J_c_point,
                              "delta_jsd": J_c_point - J_b_point,
                              "ci_lo": float(np.percentile(d, 2.5)),
                              "ci_hi": float(np.percentile(d, 97.5)),
                              "p": boot_p_two_sided(d),
                              "p_one_sided": boot_p_one_sided(d, tail="less")})
    mean_rows = []
    for cname, arrs in per_item_delta.items():
        m = np.mean(np.vstack(arrs), axis=0)
        mean_rows.append({"condition": cname, "n_items": len(arrs),
                          "mean_jsd_base": float(np.mean(np.vstack(per_item_base[cname]).mean(axis=0))),
                          "mean_jsd_cond": float(np.mean(np.vstack(per_item_cond[cname]).mean(axis=0))),
                          "mean_delta_jsd": float(np.mean(m)),
                          "ci_lo": float(np.percentile(m, 2.5)),
                          "ci_hi": float(np.percentile(m, 97.5)),
                          "p": boot_p_two_sided(m)})
    base_j = pd.DataFrame(base_rows)
    cond_j = pd.DataFrame(cond_rows)
    mean_j = pd.DataFrame(mean_rows)
    cond_j["p_BH"] = bh_adjust(cond_j["p"].values)
    cond_j["sig"] = cond_j["p_BH"] <= 0.05
    mean_j["p_BH"] = bh_adjust(mean_j["p"].values)
    mean_j["sig"] = mean_j["p_BH"] <= 0.05
    return base_j, cond_j, mean_j


def bootstrap_gap(baseline, conditions, human, n_rep=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    hu, mb = to_dict(human), to_dict(baseline)
    prepared = {c: to_dict(v) for c, v in conditions.items()}
    cond_rows = []
    per_item_delta = {}
    for name in mb:
        codes = option_codes(name_to_qid[name])
        built = build_weights(desirability_spec[name], codes)
        if built is None:
            continue
        weights, norm = built
        w = np.array([weights[c] for c in codes], dtype=float)
        H = _resample(_cvec(hu[name], codes), rng, n_rep)
        B = _resample(_cvec(mb[name], codes), rng, n_rep)
        S_h, S_b = _S(H, w), _S(B, w)
        D_b = (S_b - S_h) / norm
        absD_b = np.abs(D_b)
        for cname, cd in prepared.items():
            if name not in cd:
                continue
            C = _resample(_cvec(cd[name], codes), rng, n_rep)
            D_c = (_S(C, w) - S_h) / norm
            d_abs = np.abs(D_c) - absD_b
            per_item_delta.setdefault(cname, []).append(d_abs)
            cond_rows.append({"item": name, "condition": cname,
                              "D_base": float(np.nanmean(D_b)),
                              "D_cond": float(np.nanmean(D_c)),
                              "delta_absD": float(np.nanmean(d_abs)),
                              "ci_lo": float(np.nanpercentile(d_abs, 2.5)),
                              "ci_hi": float(np.nanpercentile(d_abs, 97.5)),
                              "p": boot_p_two_sided(d_abs),
                              "p_one_sided": boot_p_one_sided(d_abs, tail="less")})
    mean_rows = []
    for cname, arrs in per_item_delta.items():
        m = np.nanmean(np.vstack(arrs), axis=0)
        mean_rows.append({"condition": cname, "n_items": len(arrs),
                          "mean_delta_absD": float(np.nanmean(m)),
                          "ci_lo": float(np.nanpercentile(m, 2.5)),
                          "ci_hi": float(np.nanpercentile(m, 97.5)),
                          "p": boot_p_two_sided(m)})
    cond_df = pd.DataFrame(cond_rows)
    mean_df = pd.DataFrame(mean_rows)
    cond_df["p_BH"] = bh_adjust(cond_df["p"].values)
    cond_df["sig"] = cond_df["p_BH"] <= 0.05
    mean_df["p_BH"] = bh_adjust(mean_df["p"].values)
    mean_df["sig"] = mean_df["p_BH"] <= 0.05
    return cond_df, mean_df

## Configuration

In [8]:
RESULTS_ROOT = "../Publication Results"
HUMAN_CSV    = "../data/2020 ANES_test.csv"

MODEL_DIRS = {
    "Llama-3.1-8B":  "8B-Llama-Stochastic",
    "Llama-3.1-70B": "3170B-Llama-Stochastic",
    "GPT-4.1-mini":  "Gpt4.1-Mini",
}

REPLICATE_DIR = "main_mq_results"                  # original questions (baseline)
REFORM_DIR    = "main_mq_reform_results"           # full reformulation (the bundle)
THIRD_DIR     = "main_mq_reform_third_results"  # third-person framing ONLY

FILE_STEM = "full_results_2020_"                    # + QID + ".csv"

In [9]:
def _prefix(model_dir, cond_dir):
    return os.path.join(RESULTS_ROOT, model_dir, cond_dir, FILE_STEM)

def load_model(model_dir):
    replicate = get_counts(_prefix(model_dir, REPLICATE_DIR), FOUR_QIDS)
    reform    = get_counts(_prefix(model_dir, REFORM_DIR),    FOUR_QIDS)
    third     = get_counts(_prefix(model_dir, THIRD_DIR),     FOUR_QIDS)
    human     = get_human_counts(HUMAN_CSV, FOUR_QIDS)
    return replicate, reform, third, human

THIRD = "Third-person"
FULL  = "Reformulated (full)"

results = {}
for label, mdir in MODEL_DIRS.items():
    replicate, reform, third, human = load_model(mdir)
    conditions = {THIRD: third, FULL: reform}

    rows_third  = gap_analysis(replicate, third,  human)   # point estimates
    rows_reform = gap_analysis(replicate, reform, human)

    base_j, cond_j, mean_j = bootstrap_jsd(replicate, conditions, human)
    cond_d, mean_d         = bootstrap_gap(replicate, conditions, human)

    # sanity: JSD bootstrap point estimates == deterministic function
    ref = jsd_divergence_between_sets(replicate, human)
    dmax = max(abs(ref[r["item"]] - r["jsd_base"]) for _, r in base_j.iterrows())
    assert dmax < 1e-9, f"{label}: JSD point mismatch {dmax:.2e}"

    results[label] = dict(rows_third=rows_third, rows_reform=rows_reform,
                          cond_j=cond_j, mean_j=mean_j,
                          cond_d=cond_d, mean_d=mean_d)
    print(f"{label:14s} loaded  (JSD point-check {dmax:.1e})")
print("\nall models loaded")

/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_49236/3304518636.py:22: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  anes_df = pd.read_csv(anes_csv)


Llama-3.1-8B   loaded  (JSD point-check 8.3e-17)


/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_49236/3304518636.py:22: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  anes_df = pd.read_csv(anes_csv)


Llama-3.1-70B  loaded  (JSD point-check 1.7e-16)
GPT-4.1-mini   loaded  (JSD point-check 5.6e-17)

all models loaded


/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_49236/3304518636.py:22: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  anes_df = pd.read_csv(anes_csv)


## Third-person framing vs Replicate — per item

In [10]:
def sig_mark(sig):
    return "" if sig else "ns"

def print_third_table(label, R):
    cj = R["cond_j"]; cd = R["cond_d"]
    jt = cj[cj.condition == THIRD].set_index("item")
    dt = cd[cd.condition == THIRD].set_index("item")
    order = [r["item"] for r in R["rows_third"]]
    print(f"\n{label} — third-person framing only (baseline -> third-person)")
    print(f"  {'item':<18}{'ΔJSD':>9}{'':<3}{'Δ|D|':>9}{'':<3}"
          f"{'JSD CI':>20}")
    for it in order:
        j = jt.loc[it]
        dj = f"{j['delta_jsd']:+.3f}{sig_mark(j['sig']):<3}"
        ci = f"[{j['ci_lo']:+.3f},{j['ci_hi']:+.3f}]"
        if it in dt.index:
            d = dt.loc[it]
            dd = f"{d['delta_absD']:+.3f}{sig_mark(d['sig']):<3}"
        else:
            dd = f"{'—':>9}{'':<3}"
        print(f"  {it:<18}{dj:>12}{dd:>12}{ci:>20}")
    mj = R["mean_j"]; md_ = R["mean_d"]
    mjr = mj[mj.condition == THIRD].iloc[0]
    mdr = md_[md_.condition == THIRD].iloc[0]
    print(f"  {'MEAN':<18}{mjr['mean_delta_jsd']:+.3f}{sig_mark(mjr['sig']):<3}"
          f"{'':>4}{mdr['mean_delta_absD']:+.3f}{sig_mark(mdr['sig']):<3}")
    print(f"  (JSD n={int(mjr['n_items'])}, |D| n={int(mdr['n_items'])})")

for label, R in results.items():
    print_third_table(label, R)


Llama-3.1-8B — third-person framing only (baseline -> third-person)
  item                   ΔJSD        Δ|D|                 JSD CI
  Current Economy      -0.105           —        [-0.114,-0.096]
  Refugee Allowing     -0.044      -0.084        [-0.052,-0.037]
  Income Inequality    -0.035      -0.046        [-0.045,-0.024]
  Race Diversity       -0.074      -0.102        [-0.089,-0.058]
  MEAN              -0.064       -0.077   
  (JSD n=4, |D| n=3)

Llama-3.1-70B — third-person framing only (baseline -> third-person)
  item                   ΔJSD        Δ|D|                 JSD CI
  Current Economy      +0.003ns         —        [-0.006,+0.011]
  Refugee Allowing     +0.002ns    -0.075        [-0.003,+0.007]
  Income Inequality    +0.003ns    +0.018        [+0.000,+0.006]
  Race Diversity       +0.074      -0.297        [+0.064,+0.084]
  MEAN              +0.020       -0.118   
  (JSD n=4, |D| n=3)

GPT-4.1-mini — third-person framing only (baseline -> third-person)
  item        

In [11]:
def tidy_rows():
    out = []
    for label, R in results.items():
        for _, r in R["cond_j"].iterrows():
            out.append(dict(model=label, item=r["item"], condition=r["condition"],
                            metric="delta_jsd", value=r["delta_jsd"],
                            ci_lo=r["ci_lo"], ci_hi=r["ci_hi"],
                            p_BH=r["p_BH"], sig=bool(r["sig"])))
        for _, r in R["cond_d"].iterrows():
            out.append(dict(model=label, item=r["item"], condition=r["condition"],
                            metric="delta_absD", value=r["delta_absD"],
                            ci_lo=r["ci_lo"], ci_hi=r["ci_hi"],
                            p_BH=r["p_BH"], sig=bool(r["sig"])))
    return pd.DataFrame(out)

tidy = tidy_rows()
tidy.to_csv("third_person_ablation_ANES2020.csv", index=False)
print("wrote third_person_ablation_ANES2020.csv  ({} rows)".format(len(tidy)))
tidy.head(12)

wrote third_person_ablation_ANES2020.csv  (42 rows)


,model,item,condition,metric,value,ci_lo,ci_hi,p_BH,sig
0,Llama-3.1-8B,Current Economy,Third-person,delta_jsd,-0.104857,-0.113632,-0.095751,0.000267,True
1,Llama-3.1-8B,Current Economy,Reformulated (full),delta_jsd,-0.134253,-0.143119,-0.125299,0.000267,True
2,Llama-3.1-8B,Refugee Allowing,Third-person,delta_jsd,-0.044388,-0.051611,-0.037407,0.000267,True
3,Llama-3.1-8B,Refugee Allowing,Reformulated (full),delta_jsd,-0.026718,-0.032232,-0.021408,0.000267,True
4,Llama-3.1-8B,Income Inequality,Third-person,delta_jsd,-0.034642,-0.045085,-0.024165,0.000267,True
5,Llama-3.1-8B,Income Inequality,Reformulated (full),delta_jsd,-0.001405,-0.011365,0.008249,0.774723,False
6,Llama-3.1-8B,Race Diversity,Third-person,delta_jsd,-0.073663,-0.089128,-0.057905,0.000267,True
7,Llama-3.1-8B,Race Diversity,Reformulated (full),delta_jsd,-0.014272,-0.024397,-0.003776,0.010970,True
8,Llama-3.1-8B,Refugee Allowing,Third-person,delta_absD,-0.084253,-0.094468,-0.074067,0.000200,True
9,Llama-3.1-8B,Refugee Allowing,Reformulated (full),delta_absD,-0.055368,-0.064510,-0.045947,0.000200,True
